In [3]:
import numpy as np
import cv2
import os

# 1. Загрузка и сохранение исходного изображения
img = cv2.imread('sar_1.jpg', cv2.IMREAD_GRAYSCALE).astype(np.float32)
np.savetxt('image.txt', img, fmt='%d')

# 2. Вейвлет-преобразование Хаара
def haar_transform(image):
    rows, cols = image.shape
    # Делаем размеры четными
    rows = rows - rows % 2
    cols = cols - cols % 2
    image = image[:rows, :cols]
    
    # Преобразование по строкам
    temp = np.zeros((rows, cols))
    for i in range(rows):
        for j in range(0, cols, 2):
            a = image[i, j]
            b = image[i, j+1]
            temp[i, j//2] = (a + b) / 2
            temp[i, j//2 + cols//2] = (a - b) / 2
    
    # Преобразование по столбцам
    result = np.zeros((rows, cols))
    for j in range(cols):
        for i in range(0, rows, 2):
            a = temp[i, j]
            b = temp[i+1, j]
            result[i//2, j] = (a + b) / 2
            result[i//2 + rows//2, j] = (a - b) / 2
    
    # Разделение на поддиапазоны
    LL = result[:rows//2, :cols//2]
    LH = result[rows//2:, :cols//2]
    HL = result[:rows//2, cols//2:]
    HH = result[rows//2:, cols//2:]
    
    return LL, LH, HL, HH

LL, LH, HL, HH = haar_transform(img)

# 3. Квантование высокочастотных компонент (4 уровня)
def quantize(coeffs):
    min_val = np.min(coeffs)
    max_val = np.max(coeffs)
    step = (max_val - min_val) / 3 if max_val > min_val else 1
    quantized = np.floor((coeffs - min_val) / step).astype(int)
    quantized = np.clip(quantized, 0, 3)
    return quantized, min_val, step

LH_q, LH_min, LH_step = quantize(LH)
HL_q, HL_min, HL_step = quantize(HL)
HH_q, HH_min, HH_step = quantize(HH)

# 4. RLE кодирование
def rle_encode(matrix):
    flat = matrix.flatten()
    encoded = []
    current = flat[0]
    count = 1
    
    for i in range(1, len(flat)):
        if flat[i] == current:
            count += 1
        else:
            encoded.append((int(current), int(count)))
            current = flat[i]
            count = 1
    encoded.append((int(current), int(count)))
    return encoded

LH_rle = rle_encode(LH_q)
HL_rle = rle_encode(HL_q)
HH_rle = rle_encode(HH_q)

# Сохранение в файл
with open('wavelet_data.txt', 'w') as f:
    # LL компонента
    for row in LL:
        f.write(' '.join(f'{x:.2f}' for x in row) + '\n')
    f.write('\n')
    
    # LH компонента с RLE
    for val, count in LH_rle:
        f.write(f'{val} {count}\n')
    f.write('\n')
    
    # HL компонента с RLE
    for val, count in HL_rle:
        f.write(f'{val} {count}\n')
    f.write('\n')
    
    # HH компонента с RLE
    for val, count in HH_rle:
        f.write(f'{val} {count}\n')

# 5. Сравнение объемов памяти
original_size = img.nbytes
compressed_size = os.path.getsize('wavelet_data.txt')

print("РЕЗУЛЬТАТЫ:")
print(f"Исходное изображение: {original_size} байт")
print(f"После преобразования: {compressed_size} байт")
print(f"Коэффициент сжатия: {original_size/compressed_size:.2f}")
print(f"Экономия: {(1 - compressed_size/original_size)*100:.1f}%")

РЕЗУЛЬТАТЫ:
Исходное изображение: 3240000 байт
После преобразования: 1407082 байт
Коэффициент сжатия: 2.30
Экономия: 56.6%
